In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer

/opt/conda/envs/DFT/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/opt/conda/envs/DFT/lib/python3.10/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


In [2]:
model_id = "/opt/data/private/gwj/GFT/DFT/output/DFT/global_step_2150"

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype="auto",
    device_map="cuda:0"
)

tokenizer = AutoTokenizer.from_pretrained(model_id)

Skipping import of cpp extensions due to incompatible torch version 2.6.0+cu124 for torchao version 0.14.0         Please see GitHub issue #2919 for more info
/opt/conda/envs/DFT/lib/python3.10/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.


In [3]:
qwen25_math_cot = (
    "<|im_start|>system\nPlease reason step by step, and put your final answer within \\boxed{{}}.<|im_end|>\n"
    "<|im_start|>user\n{input}<|im_end|>\n"
    "<|im_start|>assistant\n",
    "{output}",
    "\n\n",
)

In [4]:
question = "Find the value of $x$ that satisfies the equation $4x+5 = 6x+7$."

In [5]:
format_input = qwen25_math_cot[0].format(input=question)

In [6]:
format_input

'<|im_start|>system\nPlease reason step by step, and put your final answer within \\boxed{}.<|im_end|>\n<|im_start|>user\nFind the value of $x$ that satisfies the equation $4x+5 = 6x+7$.<|im_end|>\n<|im_start|>assistant\n'

In [7]:
model_inputs = tokenizer([format_input], return_tensors="pt").to('cuda:0')

In [8]:
model_inputs

{'input_ids': tensor([[151644,   8948,    198,   5501,   2874,   3019,    553,   3019,     11,
            323,   2182,    697,   1590,   4226,   2878,   1124,  79075,  46391,
         151645,    198, 151644,    872,    198,   9885,    279,    897,    315,
            400,     87,      3,    429,  67901,    279,  23606,    400,     19,
             87,     10,     20,    284,    220,     21,     87,     10,     22,
          12947, 151645,    198, 151644,  77091,    198]], device='cuda:0'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1]], device='cuda:0')}

In [9]:
generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=512
)

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


In [10]:
generated_ids = [
    output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
]

response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

In [11]:
response

'To solve the equation $4x + 5 = 6x + 7$, we follow these steps:\n\n1. Subtract $4x$ from both sides to isolate the $x$ terms on one side:\n   \\[\n   4x + 5 - 4x = 6x + 7 - 4x\n   \\]\n   Simplifying, we get:\n   \\[\n   5 = 2x + 7\n   \\]\n\n2. Subtract $7$ from both sides to isolate the term with $x$:\n   \\[\n   5 - 7 = 2x + 7 - 7\n   \\]\n   Simplifying, we get:\n   \\[\n   -2 = 2x\n   \\]\n\n3. Divide both sides by $2$ to solve for $x$:\n   \\[\n   \\frac{-2}{2} = \\frac{2x}{2}\n   \\]\n   Simplifying, we get:\n   \\[\n   x = -1\n   \\]\n\nTherefore, the value of $x$ that satisfies the equation is $\\boxed{-1}$.'